# Soup Kernel Challenge: all T4 measurements in one go

Settings (right panel): **Accelerator: GPU T4 x2**, **Internet: On**. Put the repository URL in the first code cell, then **Run All** and do not touch the GPU meanwhile. About 40 minutes.

The last cell prints one report; the same text and a zip of the raw JSON files are saved in the notebook's Output (`MEASUREMENT_REPORT.txt`, `measurement_bundle.zip`).


In [ ]:
REPO_URL = "https://github.com/<you>/<your-repo>.git"  # your repository
COMMIT = ""  # the commit you submit; empty = the default branch

In [ ]:
import os, subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print(gpus)
assert "T4" in gpus, "Switch the accelerator to GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest

In [ ]:
import base64, os, shutil, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("No GITHUB_TOKEN secret, cloning without it:", type(exc).__name__)
auth = []
if token:  # the token goes in a header for this one command: never in a URL, a file or the output
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)  # always a fresh copy of your code
clone = subprocess.run(["git", *auth, "clone", "-q", REPO_URL, "/kaggle/working/repo"],
                       capture_output=True, text=True)
if clone.returncode:
    raise SystemExit("git clone failed: " + clone.stderr)
if COMMIT:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
!cd /kaggle/working/repo && python tools/run_all_measurements.py

In [ ]:
print(open('/kaggle/working/MEASUREMENT_REPORT.txt', encoding='utf-8').read())